---
---
# TUTORIAL 6: ESTIMATING PROBABILITIES BY SIMULATION

In lecture we met the **relative frequency interpretation of probability**: the probability of an event is the proportion of times it occurs in a very long series of repetitions of a random process. This gives us a practical tool: **if we can't (or don't want to) calculate a probability, we can estimate it by simulating the random process many times and computing the relative frequency of the event.**

In this tutorial you will use that tool on two problems:

1. **Snake eyes:** what's the probability of rolling two 1s with a pair of dice? (We can check our answer by counting, so this builds trust in simulation.)
2. **The Monty Hall problem:** a famous puzzle whose answer surprises almost everyone. (Here, simulation helps us *discover* the answer.)

By the end of this tutorial you should be able to:

* simulate a random process using `numpy`,
* estimate the probability of an event by its relative frequency,
* explain how the quality of the estimate changes with the number of repetitions, and
* write a short function that simulates a multi-step random process, and then repeat it with a loop.

---
---
## Setup

Run the cell below to load some familiar libraries. We'll use `numpy` for random number generation, `pandas` for tables, and `plotly.express` for plots.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

---
---
## Part 1: Snake Eyes 🎲🎲

In dice games such as craps, rolling **snake eyes** means rolling a pair of dice and having *both* dice show a 1.

Using the vocabularly from lecture:
* The **random process**: roll two fair six-sided dice (one red, one green).
* The **sample space**: all 36 possible pairs of numbers (we built this in class).
* The **event**: snake eyes, i.e., the red die shows a 1 *and* the green die shows a 1.

---

### 🧠 Step 1: What does the classical definition tell us?

Use the **classical definition** of probability to determine $\text{Pr(snake eyes)}$.

Complete the cell below to store this probability in an object called `true_prob`. We'll use it later as the "gold standard" to judge our simulation against.

**Question:** if we roll the dice 100 times, about how many snake eyes do you *expect* to see?

In [ ]:
true_prob = ____ / ____
true_prob

---
### 💻 Step 2: Simulate one roll

Let's simulate a single roll of the two dice. Fill in the blank so that `snake_eyes` is `True` when both dice show a 1 and `False` otherwise, then **run the cell over and over**. How many runs does it take before you see snake eyes?

In [ ]:
red   = ____      # fill in: roll a red die
green = ____      # fill in: roll a green die
print("Red:", red, "  Green:", green)

snake_eyes = ____      # fill in: True if both dice show a 1
print("Snake eyes?", snake_eyes)

---
### 💻 Step 3: Simulate many rolls

Now let's roll the dice `N` times at once. Complete the cell below to:

* simulate `N` rolls of the green die (the red die is done for you), then
* define the event `snake_eyes`, which is `True` for each roll in which both dice show a 1.

Recall that for an array of `True`/`False` values, `.sum()` counts the `True`s and `.mean()` gives the *proportion* of `True`s. That proportion is the **relative frequency** of snake eyes.

Run the cell at least five times and consider how much the estimates vary from run to run. Are they close to what you'd expect based on the calculation in Step 1?

In [ ]:
N = 100

red   = ____                                    # fill in: N rolls of the red die
green = ____                                    # fill in: N rolls of the green die
snake_eyes = ____                               # fill in: True for each roll that is snake eyes

print("Number of snake eyes:", snake_eyes.sum())
print("Relative frequency of snake eyes:", snake_eyes.mean())

---
### 💻 Step 4: What happens as we roll more?

The Law of Large Numbers says that as the number of repetitions grows, the relative frequency should get closer to the true probability. Let's check. 

Complete the loop below. For each value of `N`, it should simulate `N` rolls of two dice and record the relative frequency of snake eyes. Then run the second cell to tabulate the results next to the true probability from Step 1.

In [119]:
N_values = [10, 100, 1000, 10000, 100000, 1000000]
estimates = []

for N in N_values:
    ____ # fill in: roll dice and check whether snake eyes
    estimates.append(snake_eyes.mean())

In [ ]:
estimate_table = pd.DataFrame({"Number of rolls": N_values,
                               "Estimate of Pr(snake eyes)": estimates})
estimate_table["Absolute error"] = (estimate_table["Estimate of Pr(snake eyes)"] - 1/36).abs()
estimate_table

---
### 💻 Step 5: Watch the probability emerge

In lecture we plotted the *running* relative frequency of heads in a series of coin flips. Let's do the same for snake eyes. For each of five runs, we roll the dice 5000 times and, after every roll, recompute the relative frequency of snake eyes *so far* using `np.cumsum()`.

Simply run the code below to run the simulations and construct the plot.

In [121]:
N = 5000
roll_numbers = np.arange(1, N + 1)
running = pd.DataFrame({"Roll number": roll_numbers})

for run in ["Run 1", "Run 2", "Run 3", "Run 4", "Run 5"]:
    red   = np.random.randint(low = 1, high = 7, size = N)
    green = np.random.randint(low = 1, high = 7, size = N)
    snake_eyes = (red==1) & (green==1)
    running[run] = np.cumsum(snake_eyes) / roll_numbers       # relative frequency after each roll

In [ ]:
fig = px.line(running, x = "Roll number", y = ["Run 1", "Run 2", "Run 3", "Run 4", "Run 5"],
              labels = {"value": "Relative frequency of snake eyes", "variable": ""},
              title = "Relative Frequency of Snake Eyes as the Number of Rolls Grows",
              range_y = [0, 0.15], height = 500)
fig.add_hline(y = true_prob, line_dash = "dash", line_color = "black",
              annotation_text = "Pr(snake eyes) = 1/36", annotation_position = "top right")
fig.show()

**Questions:** 

* What do we see early on versus late in the repetitions?
* Compare this to the coin-flip plot from lecture. In that plot, the relative frequencies converge to 0.5 fairly quickly. Does it take *more* or *fewer* rolls for the runs to converge to 1/36 here? Why might a **rare event** take longer to "settle down"?

---
---
## Part 2: The Monty Hall Problem 🚪🚪🚪

Monty Hall was the host of the classic game show *Let's Make a Deal*. The following puzzle is named in his honour.
    <div style="text-align: center;">
    <img src="https://upload.wikimedia.org/wikipedia/en/c/cf/LMADlogo2011.jpg" alt="Monty Hall" width="450"/>
    </div>


### The Game

* There are **three doors**. Behind one door is a **car** 🚗. Behind each of the other two is a **goat** 🐐.
* You pick one door (say, Door 1). You don't open it yet.
* Monty, who **knows where the car is**, now opens a *different* door that reveals a **goat**. 
    * He never opens the door you picked.
    * He never opens the door hiding the car.
    * If he has two goat doors to choose from, he picks one at random.
* Monty then asks: ***"Do you want to stay with your original door, or switch to the other unopened door?"***
* Finally, your final door is opened. If the car is behind it, you win!

[PLAY THE GAME HERE](https://montyhall.io/)


### The Question

> **Is it better to *stay*, to *switch*, or does it make no difference?**

In other words, what are $\text{Pr(win the car if you stay)}$ and $\text{Pr(win the car if you switch)}$? Is one bigger? Are they the same?

### 🧠 What do you think?

Before touching any code, decide on your gut answer. Which is it?

* (A) Staying is better.
* (B) Switching is better.
* (C) It makes no difference. 

---
### 🗣️ Step 1: Describe the random process

Let's use the vocabulary from lecture to organize our thinking.

* **Random process:** one game of Monty Hall.
* **Outcome:** which door hides the car, which door the contestant picks, and which door Monty opens.
* **Event:** the contestant wins the car. (Whether that happens depends on the strategy, stay or switch.)

To estimate probabilities by relative frequency, we need to be able to **play one game on the computer**, and then play it many times. 

---

### 💻 Step 2: Simulate one game

Below is a function called `play_monty_hall()` which plays the game. It takes one input, `switch`, which is `True` if the contestant switches and `False` if they stay. It plays one game and returns `True` if the contestant **wins the car** and `False` otherwise.

The details of the function are *optional* for you to understand.

In [159]:
def play_monty_hall(switch):
    doors = [1, 2, 3]

    # 1. The car is behind a random door, and the contestant picks a random door.
    car  = np.random.choice(doors)
    pick = np.random.choice(doors)

    # 2. Monty opens a door that is neither the contestant's pick NOR the door hiding the car.
    monty_options = []
    for door in doors:
        if (door != car) & (door != pick):           
            monty_options.append(door)
    opened = np.random.choice(monty_options)

    # 3. The contestant either stays, or switches to the one remaining closed door.
    final_pick = pick
    if switch:
        for door in doors:
            if (door != pick) & (door != opened):          
                final_pick = door

    # 4. Return True if the contestant wins the car
    return final_pick == car

Test this function by playing 10 games with each strategy. Recall, the function returns `True` (won the car) or `False` (won a goat).

In [165]:
# Switch


In [164]:
# Stay


**Question:** Which strategy seems more promising?

---
### 💻 Step 3: Simulate many games

Now we use the relative frequency interpretation. Play `N` games with the *stay* strategy and `N` games with the *switch* strategy, and record whether each game was won. The relative frequency of wins estimates the probability of winning.

In [ ]:
N = 10000
stay_results   = []
switch_results = []

# Write a for loop that plays the Monty Hall game N times with each strategy

print("Pr(win the car if you stay)   is approximately", np.mean(stay_results))
print("Pr(win the car if you switch) is approximately", np.mean(switch_results))

**Questions:**

* Is this what you predicted? 
* What happens if you change `N` to 100? To 1000? How does the run-to-run wobble of your estimates change? (Where have we seen that before?) 

---

### 💻 Step 4: Watch the probabilities emerge

As with snake eyes, we can plot the *running* relative frequency of wins for each strategy to watch the Law of Large Numbers at work. Using the `stay_results` and `switch_results` you just simulated, fill in the blanks to compute the running relative frequency after each game.

In [ ]:
game_numbers = np.arange(1, N + 1)

stay_running   = ____               # fill in: running relative frequency of wins when staying
switch_running = ____               # fill in: running relative frequency of wins when switching

monty = pd.DataFrame({"Game number": game_numbers,
                      "Stay": stay_running,
                      "Switch": switch_running})

In [ ]:
fig = px.line(monty, x = "Game number", y = ["Stay", "Switch"],
              labels = {"value": "Relative frequency of winning the car", "variable": "Strategy"},
              title = "Monty Hall: Relative Frequency of Winning, by Strategy",
              range_y = [0, 1], height = 500)
fig.add_hline(y = 1/3, line_dash = "dash", line_color = "black",
              annotation_text = "1/3", annotation_position = "bottom right")
fig.add_hline(y = 2/3, line_dash = "dash", line_color = "black",
              annotation_text = "2/3", annotation_position = "top right")
fig.show()

The simulation suggests that **switching wins about twice as often as staying** and in particular: $$\text{Pr(win if switch)} \approx 2/3 ~~ \text{and} ~~ \text{Pr(win if stay)} \approx 1/3.$$ 

**But *why*?**

---
### 🧠 Step 5: Why does switching win?

Once Monty opens a door, two doors remain. Why isn't the chance of winning 50/50? This is the common misconception.

We're interested in $\text{Pr(win the car if you stay)}$ and $\text{Pr(win the car if you switch)}$.

What do these events say about our *initial selection*?
* If we switch and win a car, it means we initially selected a goat-door.
    * What's the probability of initially selecting a goat-door?
* If we stay and win a car, it means we initially selected a car-door.
    * What's the probability of initially selecting a car-door?

---
---